# 🔄 Flujo 2: Deduplicación
### Quédate con la última
### versión de cada registro

Típico de un CDC o de
cargas repetidas.

In [ ]:
from pyspark.sql import (
    SparkSession, functions as F,
    Window)

spark = (
    SparkSession.builder
    .appName('deduplicacion')
    .getOrCreate()
)

## 1️⃣ El problema
### 'c1' aparece 3 veces

Solo vale la fila más
reciente de cada cliente.

In [ ]:
clientes = spark.createDataFrame([
  ('c1','2025-01-01','Básico'),
  ('c1','2025-03-01','Premium'),
  ('c1','2025-02-01','Básico'),
  ('c2','2025-01-10','Free'),
  ('c2','2025-04-01','Premium'),
], ['cliente','actualizado','plan'])

clientes = clientes.withColumn(
  'actualizado',
  F.to_date('actualizado'))

clientes.show()

## 2️⃣ La solución: Window
### Numeramos por fecha ↓

Nos quedamos con la fila
número 1 (la más nueva).

In [ ]:
ventana = (
  Window
  .partitionBy('cliente')
  .orderBy(F.col('actualizado').desc())
)

ultimos = (
  clientes
  .withColumn('rn',
    F.row_number().over(ventana))
  .filter(F.col('rn') == 1)
  .drop('rn')
)

ultimos.show()

## 3️⃣ ¿row_number o rank?
### Para deduplicar:
### SIEMPRE row_number

row_number → 1 fila/clave.
rank → deja empates.

In [ ]:
print("Antes:", clientes.count(), "filas")
print("Después:", ultimos.count(), "filas")
print("1 registro por cliente ✅")

spark.stop()